In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from anthropic import Anthropic
from random import randint

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")
    
MODEL = "gpt-4o-mini"
MODEL_TRANSLATE ="claude-3-haiku-20240307"

openai = OpenAI()
translation_client = Anthropic()

In [ ]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

## Tools

With tools, you can write a function, and have the LLM call that function as part of its response.


In [ ]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

def make_booking(destination_city):
    print(f"Your booking for {destination_city} has been confirmed")
    confirmation_number = randint(100000, 999999)
    return str(confirmation_number)  

In [ ]:
# Test
get_ticket_price("London")
make_booking("Tokyo")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

booking_function = {
    "name": "make_booking",
    "description": "Make a booking or reservation for the destination city or cities. Use this when the user asks to book, make a booking or bookings, reserve, or confirm a booking or bookings."
    " If more than one destination city is mentioned, call this function for each destination city for which a booking is requested"
    " If the prices (of tickets or trips) to more than one destination city is requested in the last user message, and bookings are also requested, call this function for each city"
    " for which prices are requested. For example 'Give prices to Paris and London and book both trips.', call this function for both Paris and London."
    " If no destination is specified, use the most recently discussed destination from the conversation.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to book a flight to. This can be omitted if the destination should be inferred from conversation context.",
            },
        },
        "required": [],  # Made destination_city optional
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}, {"type": "function", "function": booking_function}]

In [ ]:
def find_last_destination_city(messages):
    """Find the last mentioned destination city in conversation context"""
    for msg in reversed(messages):
        if msg.get('role') == 'tool':
            try:
                tool_content = json.loads(msg.get('content', '{}'))
                if 'destination_city' in tool_content:
                    return tool_content['destination_city']
            except json.JSONDecodeError:
                continue
        elif msg.get('role') in ['user', 'assistant']:
            # Simple keyword extraction from conversation
            content = msg.get('content', '').lower()
            # You could enhance this with more sophisticated city detection
            common_cities = ['paris', 'london', 'tokyo', 'new york', 'berlin', 'madrid', 'rome', 'amsterdam']
            for city in common_cities:
                if city in content:
                    return city.title()
    return None

In [ ]:
def handle_tool_calls(message, messages):
    """Handle multiple tool calls from the model
     Returns a list of tool responses and a list of cities processed
     message: The user message triggering the tool calls
     messages: The full conversation history without the first argument - message
    """
    tool_responses = []
    cities = []
    
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = get_ticket_price(city)
            response = {
                "role": "tool",
                "content": json.dumps({"destination_city": city, "price": price}),
                "tool_call_id": tool_call.id
            }
            tool_responses.append(response)
            cities.append(city)
            
        elif tool_call.function.name == 'make_booking':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            
            # If no city specified, try to find from conversation context
            if not city:
                city = find_last_destination_city(messages)
            
            if not city:
                response = {
                    "role": "tool",
                    "content": json.dumps({"error": "No destination city specified or found in conversation"}),
                    "tool_call_id": tool_call.id
                }
                tool_responses.append(response)
                cities.append(None)
                continue
                
            booking_id = make_booking(city)
            response = {
                "role": "tool",
                "content": json.dumps({"destination_city": city, "booking_id": booking_id}),
                "tool_call_id": tool_call.id
            }
            tool_responses.append(response)
            cities.append(city)
        
    return tool_responses, cities



In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
from pydub import AudioSegment
from pydub.playback import play

def talker(message):
    response = openai.audio.speech.create(
      model="tts-1",
      voice="onyx",    # Also, try replacing onyx with alloy
      input=message
    )
    
    audio_stream = BytesIO(response.content)
    audio = AudioSegment.from_file(audio_stream, format="mp3")
    play(audio)

In [ ]:
def translate_to_russian(text):
    """Translate text to Russian using Claude"""
    if not text:
        return ""
    try:
        response = translation_client.messages.create(
            model="claude-3-haiku-20240307",  # Using Haiku for fast, cost-effective translation
            max_tokens=1000,
            temperature=0.3,  # Lower temperature for more consistent translation
            messages=[
                {
                    "role": "user", 
                    "content": f"Please translate the following text to Russian. Only provide the translation, no additional commentary:\n\n{text}"
                }
            ]
        )
        return response.content[0].text.strip()
    except Exception as e:
        return f"Translation error: {str(e)}"

In [ ]:
# Test the translation function
print(translate_to_russian("Hello, how are you?"))

In [ ]:
def chat_with_translation(history):
    """Your existing chat function with added Russian translation and multi-tool support"""
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # Alternative_1
    # images = []
    # Alternative_2 one image only
    image = None

    if response.choices[0].finish_reason=="tool_calls":
        # Debugging print statement
        for tool_call in response.choices[0].message.tool_calls:
            print(f"Tool called: {tool_call.function.name}")
            print(f"Arguments: {tool_call.function.arguments}")
        
        message = response.choices[0].message
        tool_responses, cities = handle_tool_calls(message, messages)
        print(f"DebugInfo, after handle_tool_calls: Tool responses: {tool_responses}")
        # Add the assistant message with tool calls
        messages.append(message)
    
        # Add all tool responses
        for tool_response in tool_responses:
            messages.append(tool_response)

        # Generate images for the valid cities
        # Alternative_1
        # for city in cities:
        #     if city:
        #         try:
        #             image = artist(city)
        #             if image:
        #                 images.append(image)
        #         except Exception as e:
        #             print(f"Error generating image for {city}: {e}")

        # Alternative_2 one image only
        if cities and cities[0]:  # Use the first city for image generation
            try:
                image = artist(cities[0])
            except Exception as e:
                print(f"Error generating image for {cities[0]}: {e}")

        # Get final response from the model
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # Comment out or delete the next line if you'd rather skip Audio for now..
    talker(reply)
    
    # Translate the AI response to Russian
    russian_translation = translate_to_russian(reply)
    
    # Return the first image if available, or None
    # Alternative_1
    # final_image = images[0] if images else None
    # Alternative_2 one image only
    final_image = image
    
    return history, final_image, russian_translation

In [ ]:
# Gradio UI setup
with gr.Blocks() as ui:
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=500, type="messages", label="Chat")
        with gr.Column(scale=1):
            image_output = gr.Image(height=250, label="Generated Image")
            # New Russian translation panel
            russian_output = gr.Textbox(
                label="Russian Translation", 
                lines=8, 
                max_lines=15,
                interactive=False,
                placeholder="Russian translation will appear here..."
            )
    
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:", scale=4)
        translate_btn = gr.Button("Translate Last Message", scale=1, variant="secondary")
    
    with gr.Row():
        clear = gr.Button("Clear Chat")
        clear_translation = gr.Button("Clear Translation")

    def do_entry(message, history):
        if message.strip():
            history = history or []
            history.append({"role": "user", "content": message})
        return "", history

    def translate_last_assistant_message(history):
        """Extract the last assistant message for translation"""
        if history:
            for msg in reversed(history):
                if msg.get("role") == "assistant":
                    return translate_to_russian(msg.get("content", ""))
        return "No assistant message found to translate."

    def clear_chat():
        return None, ""  # Clear both chatbot and russian translation

    def clear_russian():
        return ""

    # Main chat flow
    entry.submit(
        do_entry, 
        inputs=[entry, chatbot], 
        outputs=[entry, chatbot]
    ).then(
        chat_with_translation,  # Replace 'chat' with 'chat_with_translation'
        inputs=chatbot, 
        outputs=[chatbot, image_output, russian_output]
    )
    
    # Manual translation button
    translate_btn.click(
        translate_last_assistant_message,
        inputs=chatbot,
        outputs=russian_output
    )
    
    # Clear buttons
    clear.click(clear_chat, inputs=None, outputs=[chatbot, russian_output], queue=False)
    clear_translation.click(clear_russian, inputs=None, outputs=russian_output, queue=False)

ui.launch(inbrowser=True)